## Important Note — Model Selection

This project was originally designed and implemented using **Xception** (Extreme Inception) as the backbone CNN, which is a larger and more powerful architecture (22.9M parameters, 299×299 input).

During training on Google Colab, Xception achieved **~99% validation accuracy** within 7 epochs of Phase 1.

However, after over 20 failed training attempts spanning 3 days — caused by repeated GPU quota exhaustion, Colab session timeouts, and runtime disconnections — it became impractical to complete the full training pipeline with Xception.

The model was therefore switched to **MobileNetV2** (3.4M parameters, 224×224 input), a lightweight and efficient architecture that trains approximately **6× faster** while still delivering strong classification performance.

> The entire pipeline, training strategy, and evaluation methodology remain **identical** — only the base CNN was changed.

# Fruit Recognition using Deep Learning (MobileNetV2)

**Dataset:** [Fruit Recognition](https://www.kaggle.com/datasets/chrisfilo/fruit-recognition) — 44,406 labelled fruit images across 15 classes

**Approach:**
- Transfer Learning with **MobileNetV2** (pretrained on ImageNet)
- TensorFlow / Keras
- Image size: 224×224 (MobileNetV2 standard input)
- Data augmentation for better generalization

**Pipeline:**
1. Install & Import Libraries
2. Download Dataset
3. Explore Dataset Structure
4. Data Preprocessing & Augmentation
5. Build Model (Transfer Learning)
6. Train the Model
7. Evaluate Performance
8. Visualize Results
9. Save the Model
10. Inference on New Images

## 1. Install & Import Libraries

In [ ]:
!pip install -q kagglehub

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Download Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("chrisfilo/fruit-recognition")

print("Path to dataset files:", path)

## 3. Explore Dataset Structure

In [ ]:
# Explore the top-level folder structure
dataset_path = Path(path)
print(f"Dataset root: {dataset_path}")
print(f"\nTop-level contents:")
for item in sorted(dataset_path.iterdir()):
    print(f"  {'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")

In [ ]:
# Auto-detect the actual image root directory
# The dataset may have nested structure — we need the folder that contains the class folders
def find_image_root(base_path):
    """Find the directory that contains class subdirectories with images."""
    base = Path(base_path)

    # Check if current dir has subdirs with images
    subdirs = [d for d in base.iterdir() if d.is_dir()]
    if not subdirs:
        return base

    # Check if any subdir directly contains image files
    for sd in subdirs:
        files = list(sd.glob("*.jpg")) + list(sd.glob("*.jpeg")) + list(sd.glob("*.png"))
        if files:
            return base

    # If subdirs don't have images directly, check one level deeper
    for sd in subdirs:
        sub_subdirs = [d for d in sd.iterdir() if d.is_dir()]
        for ssd in sub_subdirs:
            files = list(ssd.glob("*.jpg")) + list(ssd.glob("*.jpeg")) + list(ssd.glob("*.png"))
            if files:
                return sd

    # If still not found, look for the most promising path recursively
    # Default: return the first subdir if it exists
    if len(subdirs) == 1:
        return find_image_root(subdirs[0])

    return base

image_root = find_image_root(dataset_path)
print(f"Image root directory: {image_root}")
print(f"\nClass folders:")
class_dirs = sorted([d for d in image_root.iterdir() if d.is_dir()])
for d in class_dirs:
    print(f"  {d.name}")

In [ ]:
# Count images per class (including sub-folders)
# Since each fruit folder may have sub-category sub-folders, we count all images recursively
class_counts = {}
total_images = 0

for class_dir in class_dirs:
    # Count all image files recursively
    count = 0
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        count += len(list(class_dir.rglob(ext)))
    class_counts[class_dir.name] = count
    total_images += count

print(f"Total classes: {len(class_counts)}")
print(f"Total images: {total_images}")
print(f"\nImages per class:")
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:25s} -> {cnt:6d} images")

In [ ]:
# Visualize class distribution
plt.figure(figsize=(14, 6))
classes = list(class_counts.keys())
counts = list(class_counts.values())
colors = plt.cm.viridis(np.linspace(0, 1, len(classes)))

bars = plt.bar(classes, counts, color=colors, edgecolor='black', linewidth=0.5)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.ylabel('Number of Images', fontsize=12)
plt.title('Fruit Dataset — Images per Class', fontsize=14, fontweight='bold')
plt.tight_layout()

# Add count labels on bars
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
             str(count), ha='center', va='bottom', fontsize=8)

plt.show()

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
axes = axes.flatten()

for idx, class_dir in enumerate(class_dirs[:15]):
    # Get first image (searching recursively in case of sub-folders)
    images = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        images.extend(list(class_dir.rglob(ext)))
    if images:
        img = keras.preprocessing.image.load_img(str(images[0]), target_size=(224, 224))
        axes[idx].imshow(img)
        axes[idx].set_title(class_dir.name, fontsize=11, fontweight='bold')
    axes[idx].axis('off')

# Hide any extra subplots
for idx in range(len(class_dirs), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Sample Images from Each Fruit Class', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Augmentation

Since the dataset has sub-folders within each class folder, we'll flatten the structure by creating a DataFrame mapping each image path to its parent fruit class, then use `flow_from_dataframe` for flexible loading.

In [ ]:
# Build a DataFrame of all image paths and their fruit class labels
data = []
for class_dir in class_dirs:
    class_name = class_dir.name
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        for img_path in class_dir.rglob(ext):
            data.append({
                'filepath': str(img_path),
                'label': class_name
            })

df = pd.DataFrame(data)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

print(f"Total samples: {len(df)}")
print(f"Number of classes: {df['label'].nunique()}")
print(f"\nClass distribution:\n{df['label'].value_counts()}")
df.head()

In [ ]:
# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = df['label'].nunique()
CLASS_NAMES = sorted(df['label'].unique())

print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Class names: {CLASS_NAMES}")

In [ ]:
# Split into train (80%), validation (10%), test (10%)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

In [ ]:
# Data Augmentation for training, only rescaling for validation/test
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators using flow_from_dataframe
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nClass indices: {train_generator.class_indices}")

In [ ]:
# Visualize augmented samples
sample_batch, sample_labels = next(train_generator)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()
idx_to_class = {v: k for k, v in train_generator.class_indices.items()}

for i in range(10):
    axes[i].imshow(sample_batch[i])
    label_idx = np.argmax(sample_labels[i])
    axes[i].set_title(idx_to_class[label_idx], fontsize=10, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Augmented Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Build the Model (Transfer Learning — MobileNetV2)

We use **MobileNetV2** pretrained on ImageNet as the feature extractor, then add custom classification layers on top. MobileNetV2 is lightweight and fast, making it ideal for quick training and real-time inference.

In [ ]:
# Load MobileNetV2 as feature extractor (freeze base layers)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze all base layers initially

# Build the full model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Train the Model

### Phase 1: Train only the top layers (base frozen)

In [ ]:
# Callbacks
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(
    'best_fruit_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Phase 1: Train top layers
print("=" * 60)
print("PHASE 1: Training top layers (base model frozen)")
print("=" * 60)

history1 = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

### Phase 2: Fine-tune the last layers of the base model

In [ ]:
# Phase 2: Unfreeze last 30 layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a much lower learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable layers: {sum(1 for l in model.layers if l.trainable)}")
print(f"Total trainable weights: {len(model.trainable_weights)}")

print("\n" + "=" * 60)
print("PHASE 2: Fine-tuning (last 30 base layers unfrozen)")
print("=" * 60)

# Reset callbacks for phase 2
early_stop2 = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr2 = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-8,
    verbose=1
)

checkpoint2 = callbacks.ModelCheckpoint(
    'best_fruit_model_finetuned.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

history2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[early_stop2, reduce_lr2, checkpoint2],
    verbose=1
)

## 7. Evaluate Performance

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f"\n{'='*50}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"{'='*50}")

In [ ]:
# Get predictions on the test set
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

# Map indices back to class names
idx_to_class = {v: k for k, v in test_generator.class_indices.items()}
class_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

# Classification Report
print("\nClassification Report:")
print("=" * 70)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## 8. Visualize Results

In [ ]:
# Combine training histories from both phases
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history.keys():
        combined[key] = h1.history[key] + h2.history[key]
    return combined

history = combine_histories(history1, history2)

# Plot Training & Validation Accuracy and Loss
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
axes[0].plot(history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].axvline(x=len(history1.history['accuracy'])-1, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[1].axvline(x=len(history1.history['loss'])-1, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted', fontsize=13)
plt.ylabel('True', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions on test images (correct and incorrect)
test_generator.reset()
sample_batch, sample_labels_batch = next(test_generator)
sample_preds = model.predict(sample_batch, verbose=0)

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

for i in range(20):
    axes[i].imshow(sample_batch[i])
    true_label = idx_to_class[np.argmax(sample_labels_batch[i])]
    pred_label = idx_to_class[np.argmax(sample_preds[i])]
    confidence = np.max(sample_preds[i]) * 100

    color = 'green' if true_label == pred_label else 'red'
    axes[i].set_title(
        f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)",
        fontsize=9, fontweight='bold', color=color
    )
    axes[i].axis('off')

plt.suptitle('Test Set Predictions (Green=Correct, Red=Wrong)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Save the Model

In [ ]:
# Save the final model in Keras format
model.save('fruit_recognition_model.keras')
print("Model saved as 'fruit_recognition_model.keras'")

# Save class names mapping for inference
import json
class_mapping = train_generator.class_indices
with open('class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)
print("Class mapping saved as 'class_mapping.json'")
print(f"\nClass mapping: {class_mapping}")

In [ ]:
# Download model files from Colab (run this cell in Colab)
from google.colab import files

files.download('fruit_recognition_model.keras')
files.download('class_mapping.json')
print("Download started!")

## 10. Inference on New Images

Upload any fruit image and the model will predict its class.

In [ ]:
def predict_fruit(image_path, model, class_indices):
    """Predict the fruit class for a given image."""
    idx_to_class = {v: k for k, v in class_indices.items()}

    # Load and preprocess image
    img = keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    img_array = keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model.predict(img_array, verbose=0)[0]
    top_3_idx = np.argsort(predictions)[-3:][::-1]

    # Display
    plt.figure(figsize=(6, 6))
    plt.imshow(keras.preprocessing.image.load_img(image_path, target_size=(224, 224)))
    plt.axis('off')

    title = f"Prediction: {idx_to_class[top_3_idx[0]]} ({predictions[top_3_idx[0]]*100:.1f}%)"
    plt.title(title, fontsize=14, fontweight='bold', color='green')
    plt.show()

    print("\nTop 3 Predictions:")
    for i, idx in enumerate(top_3_idx):
        print(f"  {i+1}. {idx_to_class[idx]:20s} — {predictions[idx]*100:.2f}%")

    return idx_to_class[top_3_idx[0]]

In [ ]:
# Test inference on a random image from the test set
random_idx = np.random.randint(0, len(test_df))
random_image_path = test_df.iloc[random_idx]['filepath']
true_label = test_df.iloc[random_idx]['label']

print(f"True label: {true_label}")
predicted = predict_fruit(random_image_path, model, train_generator.class_indices)

In [ ]:
# Upload your own image to test (Colab only)
from google.colab import files

print("Upload a fruit image to classify:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nPredicting: {filename}")
    predict_fruit(filename, model, train_generator.class_indices)

## Summary

| Item | Detail |
|------|--------|
| **Dataset** | Fruit Recognition — 44,406 images, 15 classes |
| **Model** | MobileNetV2 (Transfer Learning) |
| **Input Size** | 224×224 |
| **Training** | Phase 1: Frozen base (15 epochs) → Phase 2: Fine-tune last 30 layers (10 epochs) |
| **Augmentation** | Rotation, shift, shear, zoom, flip, brightness |
| **Optimizer** | Adam (1e-3 → 1e-5) |
| **Callbacks** | EarlyStopping, ReduceLROnPlateau, ModelCheckpoint |

### How to use the saved model later:
```python
import tensorflow as tf
import json

model = tf.keras.models.load_model('fruit_recognition_model.keras')
with open('class_mapping.json') as f:
    class_mapping = json.load(f)
```

In [ ]:
# End of notebook
print("Fruit Recognition project complete!")